In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import pandas as pd
import numpy as np
import pickle
from sklearn.neighbors import NearestNeighbors

# for descriptors calculation
import datamol as dm
from molfeat.calc.descriptors import RDKitDescriptors2D, RDKitDescriptors3D
from molfeat.trans import MoleculeTransformer

from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import pandas as pd
import numpy as np
import pickle
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from molfeat_padel.calc import PadelDescriptors


from rdkit import Chem
from rdkit.Chem import AllChem

from function import embed_optimize

Failed to find the pandas get_adjustment() function to patch
Failed to patch pandas - PandasTools will have limited functionality


In [4]:
with open('X_padel_dropped.pickle', 'rb') as inp:
    X = pickle.load(inp)

In [7]:
tranpX = np.transpose(X)
xtx = np.dot(tranpX, X)
invxtx = np.linalg.pinv(xtx)

In [8]:
warning_level = 3*(X.shape[1]/X.shape[0])

In [10]:
smi = 'Cc1ccccc1'
mol = embed_optimize(smi)

In [11]:
desc_calc = PadelDescriptors()
with dm.without_rdkit_log():
    descs = pd.DataFrame(desc_calc(mol), index = desc_calc.columns).transpose()
    
with open('imputer.pickle', 'rb') as inp:
    imputer = pickle.load(inp) 

descs = pd.DataFrame(imputer.transform(descs), columns = descs.columns)

with open('features_to_drop_Padel.pickle', 'rb') as inp:
        features_to_drop = pickle.load(inp)

descs = descs.drop(columns = features_to_drop)

In [12]:
descs

,PaDEL_nAcid,PaDEL_ALogP,PaDEL_ALogp2,PaDEL_AMR,PaDEL_nB,PaDEL_nS,PaDEL_nP,PaDEL_nF,PaDEL_nI,PaDEL_AATS3m,...,PaDEL_PubchemFP871,PaDEL_PubchemFP872,PaDEL_PubchemFP873,PaDEL_PubchemFP874,PaDEL_PubchemFP875,PaDEL_PubchemFP876,PaDEL_PubchemFP877,PaDEL_PubchemFP878,PaDEL_PubchemFP879,PaDEL_PubchemFP880
0,0.0,0.642,0.412164,5.5021,0.0,0.0,0.0,0.0,0.0,34.937498,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
leverage = np.dot(np.dot(descs, invxtx), np.transpose(descs))[0][0]
print(leverage)

0.990665835763


In [17]:
with open('app_domain.pickle', 'wb') as out:
    pickle.dump({'invxtx':invxtx, 'warning':warning_level}, out)